# Module 2.1: Find a source and add related graph facts

Use this notebook when a question needs facts from both a source document and related graph records. For example, to find Chicago hotels with both a spa and a swimming pool, you need to check more than one fact for each hotel.

Start with semantic search to find the matching source. Then follow graph relationships to return the related facts and show where they came from.

**Overview**

- **Semantic search:** It finds source text with meaning similar to the question.
- **Exact-term search:** It finds source text that contains an important word or identifier.
- **Graph expansion:** It adds related hotel facts, such as amenities and ratings, to a matching source.
- **Structured filtering:** It checks clear conditions, such as whether the same hotel has a spa and a swimming pool.
- **What is checked:** This notebook checks retrieved records and their sources. It does not evaluate generated answer wording.

The optional Text2Cypher example appears after the core retrieval exercises.

## Check that Module 1 finished

Run the cells in order after you finish Module 1. The next check runs in Python, so you do not need a terminal command.

- **Read-only:** This notebook reads the graph. It does not rebuild or clear your work.
- **If the check fails:** Return to Module 1, run every cell, then restart this notebook from the top.

In [ ]:
import os
import sys
from pathlib import Path

# The shared workshop/ package lives in notebooks/. These lines find that
# directory and put it on the import path. Everything else this notebook
# needs to start is in workshop/bootstrap.py.
_here = Path.cwd().resolve()
_named = os.environ.get("WORKSHOP_NOTEBOOKS_DIR") or _here
_starts = (Path(_named).expanduser().resolve(), _here, _here / "notebooks")
for _candidate in (*_starts, *_here.parents):
    if (_candidate / "workshop" / "bootstrap.py").is_file():
        sys.path.insert(0, str(_candidate))
        break

from workshop.bootstrap import start_module

NOTEBOOKS_ROOT, REPO_ROOT, MODULE_DIR = start_module("02-connected-context")
print(f'Workshop root: {REPO_ROOT}')

In [ ]:
from IPython.display import HTML, display
from neo4j import GraphDatabase, Query, READ_ACCESS
from neo4j_graphrag.retrievers import (
    HybridRetriever,
    VectorCypherRetriever,
    VectorRetriever,
)
from neo4j_graphrag.types import RetrieverResultItem

from neo4j_graphrag.retrievers.text2cypher import extract_cypher

from workshop.aws_region import aws_region, configure_aws_region
from workshop.bedrock_providers import BedrockEmbeddings, BedrockLLM
from workshop.graph_connection import (
    graph_database,
    neo4j_auth,
    neo4j_uri,
    require_neo4j_env,
)
from workshop.graph_schema import GRAPH_SCHEMA
from workshop.hybrid_retrieval import (
    GRAPH_QUERY_EXAMPLES,
    GRAPH_QUERY_PROMPT,
    pinned_schema_text,
    search_hotel_knowledge,
)
from workshop.retrieval_contract import (
    CHUNK_FULLTEXT_INDEX,
    CHUNK_VECTOR_INDEX,
    EMBEDDING_DIMENSIONS,
)
from workshop.retrieval_setup import (
    CHICAGO_CITY,
    CHICAGO_EXCLUSION,
    CHICAGO_FILTER_QUERY,
    CHICAGO_QUALIFIER,
    CHICAGO_SOURCE_FILES,
    chicago_filter_problems,
    chicago_filter_records,
    fixture_for,
    graph_context_problems,
    report_problems,
    source_context_problems,
    source_fixture_problems,
    verify_retrieval_indexes,
)

configure_aws_region()
require_neo4j_env()
DATABASE = graph_database()
# The retrievers below come from neo4j-graphrag, and they still call
# db.index.vector.queryNodes, which Neo4j has deprecated in favor of SEARCH.
# The deprecated call belongs to the library rather than to this notebook, and
# the results are correct either way, so the server notice it raises on every
# retrieval is noise here. notifications_min_severity silences it, matching the
# drivers that hybrid_retrieval, reservation_command, and fixtures already open.
# Module 3 reads through those, so leaving this one loud made the same query
# look different depending on which file opened the connection.
driver = GraphDatabase.driver(
    neo4j_uri(), auth=neo4j_auth(), notifications_min_severity="OFF"
)
driver.verify_connectivity()
print(f'Connected to Neo4j database: {DATABASE}')

## Check that the graph has the required data

Run this cell before the retrieval examples. It checks that the Cairo and Chicago source documents, embeddings, entities, relationships, and indexes are ready.

In [ ]:
verify_retrieval_indexes(driver)
print(f'PASS  {CHUNK_VECTOR_INDEX}: online, cosine, {EMBEDDING_DIMENSIONS} dimensions')
print(f'PASS  {CHUNK_FULLTEXT_INDEX}: online over Chunk.text')

problems = source_fixture_problems(driver)
problems.extend(chicago_filter_problems(chicago_filter_records(driver)))
if problems:
    problems.append(
        'Return to the Module 1 notebook, run every cell, then rerun this '
        'notebook from the top. This check did not modify the graph.'
    )
report_problems(
    problems,
    'Cairo and Chicago source, path, field, amenity, and filter fixtures',
)

## Show the graph structure and embedding settings

This cell shows the graph structure used by the extraction pipeline and each graph query.

- **`GRAPH_SCHEMA`:** It holds the shared list of node labels and relationships used in this workshop.
- **Embedding settings:** Each text retriever uses the same 1024-dimensional Amazon Nova settings that created the chunk vectors.
- **Database:** Each query uses the configured Neo4j database.

In [ ]:
pattern_rows = ''.join(
    f'<tr><td><strong>{source}</strong></td><td>-[:{relationship}]-&gt;</td>'
    f'<td><strong>{target}</strong></td></tr>'
    for source, relationship, target in GRAPH_SCHEMA['patterns']
)
display(HTML(
    '<table><thead><tr><th>From</th><th>Relationship</th><th>To</th></tr>'
    f'</thead><tbody>{pattern_rows}</tbody></table>'
    '<p>Text provenance: <code>(:Chunk)-[:FROM_DOCUMENT]-&gt;(:Document)</code>.</p>'
    '<p>Entity provenance: <code>(:Hotel)-[:FROM_CHUNK]-&gt;(:Chunk)</code>.</p>'
))

## Set up the retrieved-context display

The next code cell defines one shared display format for every retrieval result. Every retriever prints the same fields in the same order, so you can compare results side by side.

- **Question and retriever:** Show what was asked and which retriever ran.
- **Ranking details:** Show the top-k setting, rank, and score.
- **Source text:** Show the source filename and complete `Chunk.text`.
- **Context size:** Show the size of the structured fields and source text separately.
- **Missing fields:** Show any requested fields that the result does not contain.

Each source filename comes from the graph provenance path for that result.

In [ ]:
def source_for_chunk_text(chunk):
    with driver.session(database=DATABASE, default_access_mode=READ_ACCESS) as session:
        record = session.run(
            '''
            MATCH (matched:Chunk {text: $chunk})-[:FROM_DOCUMENT]->(document:Document)
            RETURN collect(DISTINCT document.source_filename) AS source_filenames
            ''',
            chunk=chunk,
        ).single(strict=True)
    return ', '.join(record['source_filenames'])


def context_value_chars(value):
    if value is None:
        return 0
    if isinstance(value, dict):
        return sum(
            context_value_chars(key) + context_value_chars(item)
            for key, item in value.items()
        )
    if isinstance(value, (list, tuple, set)):
        return sum(context_value_chars(item) for item in value)
    return len(str(value))


def context_char_counts(structured_fields, source_text):
    structured_chars = sum(
        context_value_chars(value) for value in structured_fields.values()
    )
    return structured_chars, len(source_text)


def print_record(record, fields):
    for field in fields:
        print(f'  {field}: {record.get(field)}')


def text_result_formatter(record):
    node = record.get('node') or {}
    chunk = node.get('text') or ''
    return RetrieverResultItem(
        content=chunk,
        metadata={
            'score': record.get('score'),
            'source_filename': source_for_chunk_text(chunk),
        },
    )


embedder = BedrockEmbeddings(region_name=aws_region())


def context_rows(
    question,
    retriever_name,
    result,
    top_k,
    configuration,
    required_terms,
    absent_fields=(),
):
    """Print every retrieval result the same way and return the rows.

    `required_terms` maps a requested field to the text that has to appear in
    the source chunk for that field to count as present. `absent_fields` names
    fields the source text cannot supply at all, which is what graph expansion
    later adds.
    """
    print(f'Question: {question}')
    print(f'Retriever: {retriever_name}')
    print(f'Configuration: {configuration}')
    print(f'Top-k: {top_k}')
    print(f'Result count: {len(result.items)}')
    rows = []
    for rank, item in enumerate(result.items, 1):
        metadata = item.metadata or {}
        chunk = str(item.content or '')
        source_text = chunk.casefold()
        missing = [] if metadata.get('source_filename') else ['source_filename']
        missing.extend(
            field for field, term in required_terms.items()
            if term.casefold() not in source_text
        )
        missing.extend(absent_fields)
        structured_chars, source_text_chars = context_char_counts(
            {'source_filename': metadata.get('source_filename')}, chunk
        )
        row = {
            'rank': rank,
            'score': metadata.get('score'),
            'source_filename': metadata.get('source_filename'),
            'structured_context_chars': structured_chars,
            'source_text_chars': source_text_chars,
            'missing_requested_fields': missing,
            'chunk': chunk,
        }
        rows.append(row)
        score = row['score']
        score_text = 'n/a' if score is None else f'{score:.6f}'
        print(f'Rank {rank} | score={score_text} | source={row["source_filename"]}')
        print(f'Structured-field context: {structured_chars} characters')
        print(f'Source-text context: {source_text_chars} characters')
        print(f'Missing requested fields: {missing or "none"}')
        print('Complete Chunk text:')
        print(chunk)
        print()
    return rows

## 1. Find an arrival time with semantic search

This example uses `VectorRetriever` to find a source by meaning. The question says "arrival processing" instead of `Standard check-in time`, so it tests whether semantic search can find the Cairo source when the wording changes.

- **Input:** The question asks about the arrival time at AnyCompany Cairo Nile View.
- **Expected context:** The retrieved context includes the Cairo source and its supported `3:00 PM` arrival time.
- **Check:** The test checks the retrieved context only.
- **Chunk size:** Extraction keeps each hotel in one large chunk so the pipeline can build its graph. That same large chunk returns during search. A production system can split into smaller, separate retrieval chunks sized to the source material and the context budget.

In [ ]:
CAIRO_FIXTURE = fixture_for('hotel-cairo-001.txt')
vector_retriever = VectorRetriever(
    driver=driver,
    index_name=CHUNK_VECTOR_INDEX,
    embedder=embedder,
    return_properties=['text'],
    result_formatter=text_result_formatter,
    neo4j_database=DATABASE,
)
ARRIVAL_QUESTION = (
    'When does standard arrival processing begin at AnyCompany Cairo Nile View?'
)
VECTOR_TOP_K = 3
arrival_result = vector_retriever.search(
    query_text=ARRIVAL_QUESTION,
    top_k=VECTOR_TOP_K,
)
arrival_rows = context_rows(
    ARRIVAL_QUESTION,
    'VectorRetriever',
    arrival_result,
    VECTOR_TOP_K,
    f'index={CHUNK_VECTOR_INDEX}, cosine, Nova {EMBEDDING_DIMENSIONS} dimensions',
    {'supported_arrival_time': '3:00 PM'},
)
report_problems(
    source_context_problems(arrival_rows, CAIRO_FIXTURE),
    'Cairo source and supported 3:00 PM arrival time are visible.',
)

## 2. Find a postal code with hybrid search

This example compares `VectorRetriever` and `HybridRetriever` with the same top-five limit. Postal code `60611` is an exact identifier. The full-text index matches its exact characters, while semantic search only matches similar meaning, so full-text search finds it more reliably.

- **Vector signal:** The complete question supplies the semantic search input.
- **Full-text signal:** `60611` supplies the exact-term search input.
- **Hybrid ranking:** The reviewed linear ranker combines both signals with `alpha=0.2`.
- **Output:** One line per result for each retriever, giving its rank, its source filename, and whether its text contains `60611`. Both retrievers rank on the same question vector, so the ranking difference comes from the full-text term alone.
- **Check:** The test checks that the hybrid context carries the Chicago hotel, its postal code, and its cancellation policy.

In [ ]:
CHICAGO_IDENTIFIER_FIXTURE = fixture_for('hotel-chicago-001.txt')
CANCELLATION_TERM = 'at least 24 hours prior to arrival'
IDENTIFIER_QUESTION = 'What is the cancellation policy for the hotel at 60611?'
IDENTIFIER_TOP_K = 5

hybrid_retriever = HybridRetriever(
    driver=driver,
    vector_index_name=CHUNK_VECTOR_INDEX,
    fulltext_index_name=CHUNK_FULLTEXT_INDEX,
    embedder=embedder,
    return_properties=['text'],
    result_formatter=text_result_formatter,
    neo4j_database=DATABASE,
)

vector_identifier_result = vector_retriever.search(
    query_text=IDENTIFIER_QUESTION,
    top_k=IDENTIFIER_TOP_K,
)
# Both searches rank on the same question vector, so the only difference
# between the two result lists is the full-text term the hybrid search adds.
hybrid_identifier_result = hybrid_retriever.search(
    query_text='60611',
    query_vector=vector_identifier_result.metadata['query_vector'],
    top_k=IDENTIFIER_TOP_K,
    ranker='linear',
    alpha=0.2,
)


def identifier_rows(result):
    """Return the source filename and text of each result, in rank order.

    This comparison needs two fields per result, so it reads them straight
    off the result instead of going through `context_rows`. That display
    prints the complete chunk text, which examples 1 and 3 need to read and
    this ranking comparison does not.
    """
    return [
        {
            'rank': rank,
            'source_filename': (item.metadata or {}).get('source_filename'),
            'chunk': str(item.content or ''),
        }
        for rank, item in enumerate(result.items, 1)
    ]


hybrid_identifier_rows = identifier_rows(hybrid_identifier_result)
print(f'Question: {IDENTIFIER_QUESTION}')
print(f'Top-k: {IDENTIFIER_TOP_K}')
print('Vector signal: complete question. Full-text term: 60611.')
print('Hybrid ranking: linear ranker, alpha=0.2.')
for label, rows in (
    ('Vector', identifier_rows(vector_identifier_result)),
    ('Hybrid', hybrid_identifier_rows),
):
    for row in rows:
        print(
            f'{label} rank {row["rank"]} | source={row["source_filename"]} | '
            f'contains 60611: {"60611" in row["chunk"]}'
        )
report_problems(
    source_context_problems(
        hybrid_identifier_rows,
        CHICAGO_IDENTIFIER_FIXTURE,
        extra_terms=(CANCELLATION_TERM,),
    ),
    'Hybrid context contains the hotel, postal code, and cancellation policy.',
)

## 3. Find a source, then add related hotel facts

This example uses `VectorCypherRetriever` to combine semantic search with a reviewed graph query. It finds the matching source `Chunk` first, then follows graph relationships to add hotel fields and amenities.

- **Source record:** The result keeps the matching `Chunk`, its `Document`, and the semantic score.
- **Added facts:** The graph query returns the hotel name, hotel ID, guest rating, and amenities.
- **Provenance:** Each field shows the graph path that produced it.
- **Source of truth:** The graph result reflects what extraction placed in Neo4j. It is not an independent source of truth. Compare it with the source text to find missing or merged facts.

In [ ]:
VECTOR_CYPHER_QUERY = '''
MATCH (node)-[:FROM_DOCUMENT]->(document:Document)
OPTIONAL MATCH (hotel:Hotel)-[:FROM_CHUNK]->(node)
OPTIONAL MATCH (hotel)-[:OFFERS_AMENITY]->(amenity:Amenity)
WITH node, score, document, hotel, collect(DISTINCT amenity.name) AS amenities
RETURN hotel.name AS hotel_name,
       hotel.hotel_id AS hotel_id,
       hotel.guest_rating AS guest_rating,
       document.source_filename AS source_filename,
       amenities,
       node.text AS source_chunk,
       score AS semantic_score,
       CASE
           WHEN hotel IS NULL THEN ['FROM_DOCUMENT']
           ELSE ['FROM_DOCUMENT', 'FROM_CHUNK', 'OFFERS_AMENITY']
       END AS relationship_types,
       CASE
           WHEN hotel IS NULL THEN 'missing Hotel enrichment for semantic hit'
           ELSE 'complete Hotel enrichment'
       END AS graph_enrichment_status,
       {
           source_chunk: '(:Chunk)-[:FROM_DOCUMENT]->(:Document)',
           source_filename: '(:Chunk)-[:FROM_DOCUMENT]->(:Document)',
           hotel_name: '(:Hotel)-[:FROM_CHUNK]->(:Chunk)',
           hotel_id: '(:Hotel)-[:FROM_CHUNK]->(:Chunk)',
           guest_rating: '(:Hotel)-[:FROM_CHUNK]->(:Chunk)',
           amenities: '(:Hotel)-[:OFFERS_AMENITY]->(:Amenity)'
       } AS field_provenance
ORDER BY semantic_score DESC,
         CASE WHEN hotel_id IS NULL THEN 1 ELSE 0 END ASC,
         hotel_id ASC
'''

GRAPH_REQUESTED_FIELDS = (
    'hotel_name',
    'hotel_id',
    'guest_rating',
    'source_filename',
    'amenities',
)
# The fields graph expansion returns as structured data, which is what the
# comparison at the end of this cell counts against plain vector search.
GRAPH_STRUCTURED_FIELDS = (
    *GRAPH_REQUESTED_FIELDS,
    'relationship_types',
    'graph_enrichment_status',
    'field_provenance',
)
GRAPH_RECORD_FIELDS = (
    *GRAPH_REQUESTED_FIELDS,
    'semantic_score',
    'relationship_types',
    'graph_enrichment_status',
    'field_provenance',
    'missing_requested_fields',
    'structured_context_chars',
    'source_text_chars',
)
# Plain vector search returns one structured field. Everything else a reader
# sees is text they would have to parse themselves.
VECTOR_NAMED_FIELDS = ('source_filename',)


def graph_result_formatter(record):
    metadata = {
        field: record.get(field) for field in GRAPH_REQUESTED_FIELDS
    }
    metadata['amenities'] = metadata['amenities'] or []
    metadata.update(
        semantic_score=record.get('semantic_score'),
        relationship_types=record.get('relationship_types') or [],
        graph_enrichment_status=record.get('graph_enrichment_status'),
        field_provenance=record.get('field_provenance') or {},
    )
    metadata['missing_requested_fields'] = [
        field for field in GRAPH_REQUESTED_FIELDS
        if metadata.get(field) is None or metadata.get(field) == []
    ]
    return RetrieverResultItem(
        content=record.get('source_chunk') or '',
        metadata=metadata,
    )


vector_cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name=CHUNK_VECTOR_INDEX,
    retrieval_query=VECTOR_CYPHER_QUERY,
    embedder=embedder,
    result_formatter=graph_result_formatter,
    neo4j_database=DATABASE,
)
CAIRO_GRAPH_QUESTION = (
    'What amenities and guest rating does AnyCompany Cairo Nile View have?'
)
GRAPH_TOP_K = 3
graph_vector_result = vector_retriever.search(
    query_text=CAIRO_GRAPH_QUESTION,
    top_k=GRAPH_TOP_K,
)
graph_vector_rows = context_rows(
    CAIRO_GRAPH_QUESTION,
    'VectorRetriever',
    graph_vector_result,
    GRAPH_TOP_K,
    f'index={CHUNK_VECTOR_INDEX}, semantic entry before graph expansion',
    {
        'hotel_name': CAIRO_FIXTURE.hotel_name,
        'guest_rating': f'{CAIRO_FIXTURE.guest_rating}/5.0',
        'amenities': 'Hotel Amenities',
    },
    absent_fields=('hotel_id',),
)
graph_result = vector_cypher_retriever.search(
    query_text=CAIRO_GRAPH_QUESTION,
    top_k=GRAPH_TOP_K,
)

graph_records = []
print(f'Question: {CAIRO_GRAPH_QUESTION}')
print('Retriever: VectorCypherRetriever')
print(f'Configuration: index={CHUNK_VECTOR_INDEX}, top_k={GRAPH_TOP_K}, reviewed traversal')
print(f'Result count: {len(graph_result.items)}')
for rank, item in enumerate(graph_result.items, 1):
    record = dict(item.metadata or {})
    record['source_chunk'] = str(item.content or '')
    record['rank'] = rank
    structured_chars, source_text_chars = context_char_counts(
        {field: record[field] for field in GRAPH_STRUCTURED_FIELDS},
        record['source_chunk'],
    )
    record['structured_context_chars'] = structured_chars
    record['source_text_chars'] = source_text_chars
    graph_records.append(record)
    print(f'Record {rank}:')
    print_record(record, GRAPH_RECORD_FIELDS)
    print('  source_chunk:')
    print(record['source_chunk'])

missing_enrichment = [
    record for record in graph_records
    if record['graph_enrichment_status'].startswith('missing')
]
print(
    f'Graph enrichment: {len(graph_records) - len(missing_enrichment)} complete, '
    f'{len(missing_enrichment)} missing Hotel context. Semantic hits without an '
    'extracted Hotel stay visible instead of disappearing.'
)

cairo_graph = next(
    record for record in graph_records
    if record['source_filename'] == CAIRO_FIXTURE.source_filename
)
vector_cairo = next(
    row for row in graph_vector_rows
    if row['source_filename'] == CAIRO_FIXTURE.source_filename
)
requested = len(GRAPH_REQUESTED_FIELDS)
graph_named = requested - len(cairo_graph['missing_requested_fields'])
print('Context comparison:')
print(
    f'  Vector: {len(VECTOR_NAMED_FIELDS)}/{requested} named fields, '
    f'{vector_cairo["structured_context_chars"]} structured and '
    f'{vector_cairo["source_text_chars"]} source-text characters'
)
print(
    f'  Vector-Cypher: {graph_named}/{requested} named fields, '
    f'{cairo_graph["structured_context_chars"]} structured and '
    f'{cairo_graph["source_text_chars"]} source-text characters'
)
print('Extraction quality limits graph enrichment. Missing extracted relationships stay missing.')
report_problems(
    graph_context_problems(graph_records, CAIRO_FIXTURE),
    'Cairo Vector-Cypher context includes every locked field and provenance path.',
)

## 4. Find Chicago hotels with a spa and a swimming pool

This example uses a shared, fixed Cypher query to filter Chicago hotels. The query checks both amenity conditions for the same hotel.

- **Candidates:** The query checks every Chicago hotel as a candidate.
- **Qualifier:** A qualifying hotel has both a spa and a swimming pool.
- **Exclusion:** An excluded hotel is missing a spa, a swimming pool, or both.

In [ ]:
CHICAGO_QUESTION = 'Which hotels in Chicago offer both a spa and a swimming pool?'
CHICAGO_PATTERN_NAME = 'Reviewed fixed Cypher: same-hotel spa AND pool filter'
CHICAGO_CONFIGURATION = 'city predicate, reviewed two-amenity AND filter'
CHICAGO_REQUESTED_FIELDS = ('hotel_name', 'guest_rating', 'amenities', 'source_filename')
CHICAGO_RECORD_FIELDS = (
    *CHICAGO_REQUESTED_FIELDS,
    'qualifies',
    'missing_required_amenities',
    'missing_requested_fields',
    'field_provenance',
    'structured_context_chars',
    'source_text_chars',
)
CHICAGO_PROVENANCE = {
    'source_chunk': '(:Chunk)-[:FROM_DOCUMENT]->(:Document)',
    'source_filename': '(:Chunk)-[:FROM_DOCUMENT]->(:Document)',
    'hotel_name': '(:Hotel)-[:FROM_CHUNK]->(:Chunk)',
    'guest_rating': '(:Hotel)-[:FROM_CHUNK]->(:Chunk)',
    'amenities': '(:Hotel)-[:OFFERS_AMENITY]->(:Amenity)',
}


def fixed_cypher_context(record):
    record['field_provenance'] = CHICAGO_PROVENANCE
    record['missing_requested_fields'] = [
        field for field in CHICAGO_REQUESTED_FIELDS
        if record.get(field) is None or record.get(field) == []
    ]
    structured_fields = {
        field: record.get(field)
        for field in (*CHICAGO_REQUESTED_FIELDS, 'qualifies', 'missing_required_amenities')
    }
    structured_chars, source_text_chars = context_char_counts(
        structured_fields, record.get('source_chunk') or ''
    )
    record['structured_context_chars'] = structured_chars
    record['source_text_chars'] = source_text_chars
    return record


candidate_records = [
    fixed_cypher_context(record) for record in chicago_filter_records(driver)
]

print(f'Question: {CHICAGO_QUESTION}')
print(f'Pattern: {CHICAGO_PATTERN_NAME}')
print(f'Configuration: {CHICAGO_CONFIGURATION}')
print(f'Reviewed Cypher: {CHICAGO_FILTER_QUERY}')
print(f'Parameters: city={CHICAGO_CITY!r}')
print(f'Expected sources: {list(CHICAGO_SOURCE_FILES)}')
print(f'Expected qualifier: {CHICAGO_QUALIFIER}')
print(f'Expected exclusion: {CHICAGO_EXCLUSION}, which has no spa and no swimming pool')
print(f'Candidate count: {len(candidate_records)}')
for record in candidate_records:
    verdict = 'qualifies' if record['qualifies'] else 'excluded'
    print(f'Candidate: {record["hotel_name"]} ({verdict})')
    print_record(record, CHICAGO_RECORD_FIELDS)
report_problems(
    chicago_filter_problems(candidate_records),
    'Two candidates, one qualifier, and the Windward exclusion are explicit.',
)

## Optional: Generate Cypher for a flexible question

Use this optional example for a structured question that does not have a fixed query. It uses the same workshop credentials as every other cell, along with the shared model, pinned schema, and configured database. It gives each database query a 15-second timeout.

- **Generated query:** The model creates Cypher from the question.
- **Safety check:** `EXPLAIN` plans the query first. The cell runs only queries that the planner marks as read-only.
- **Output:** The cell shows the query, validation result, records, and errors.

Use a read-only Neo4j user in production. The database then enforces read-only access as well.

In [ ]:
TEXT2CYPHER_TIMEOUT_SECONDS = 15


def run_optional_text2cypher(question):
    outcome = {
        'question': question,
        'generated_cypher': '',
        'read_only_validation': 'not planned',
        'records': [],
        'result_count': 0,
        'displayed_count': 0,
        'execution_error': None,
    }
    try:
        prompt = GRAPH_QUERY_PROMPT.format(
            schema=pinned_schema_text(),
            examples=' '.join(GRAPH_QUERY_EXAMPLES),
            query_text=question,
        )
        response = BedrockLLM(region_name=aws_region()).invoke(prompt)
        cypher = extract_cypher(response.content)
        outcome['generated_cypher'] = cypher

        with driver.session(
            database=DATABASE,
            default_access_mode=READ_ACCESS,
        ) as session:
            summary = session.run(
                Query(f'EXPLAIN {cypher}', timeout=TEXT2CYPHER_TIMEOUT_SECONDS)
            ).consume()
            if summary.query_type != 'r':
                outcome['read_only_validation'] = (
                    f'rejected: query_type={summary.query_type}'
                )
                raise RuntimeError('The planner did not classify the query as read-only')
            outcome['read_only_validation'] = 'passed: EXPLAIN query_type=r'
            all_records = session.run(
                Query(cypher, timeout=TEXT2CYPHER_TIMEOUT_SECONDS)
            ).data()
            outcome['result_count'] = len(all_records)
            outcome['records'] = all_records[:25]
            outcome['displayed_count'] = len(outcome['records'])
    except Exception as exc:
        outcome['execution_error'] = f'{type(exc).__name__}: {exc}'
    return outcome


text2cypher_outcome = run_optional_text2cypher(CHICAGO_QUESTION)
for field, value in text2cypher_outcome.items():
    print(f'{field}: {value}')
print('Supporting Text2Cypher output shown. Fixed Cypher remains the acceptance path.')

## Select a retriever for the question

| Question type | Use first | Check these records | Use in this workshop |
|---|---|---|---|
| Question uses different wording | `VectorRetriever` | Ranked `Chunk` nodes, vector scores, source paths | Basic semantic search |
| Question includes an exact term | `HybridRetriever` | Exact-term hits and ranked `Chunk` nodes | Exact-term support |
| Question needs related hotel facts | `VectorCypherRetriever` | Source `Chunk`, graph fields, relationships, source paths | Connected-context search |
| Question has known, fixed conditions | Reviewed fixed Cypher | Candidate, qualifier, and exclusion records | Fixed, repeatable filter |
| Flexible structured question | Text2Cypher | Generated query, planner result, records, errors | Optional query with a safety check |
| Question needs both an exact term and related hotel facts | `HybridCypherRetriever` | Fused-score `Chunk`, graph fields, relationships, source paths | Module 3 handoff |

- **`HybridCypherRetriever`:** It runs the same vector-plus-full-text fusion shown above for `HybridRetriever`, then adds the same kind of graph traversal shown above for `VectorCypherRetriever`. This notebook does not run it directly; it lives in `workshop/hybrid_retrieval.py`.
- **Module 2 choice:** The booking question needs an exact hotel name and related named fields in one context record. That needs both the exact-term match `HybridRetriever` demonstrated and the graph traversal `VectorCypherRetriever` demonstrated. `HybridCypherRetriever` does both in one call.
- **Module 3 handoff:** The application exposes this retriever through `search_hotel_knowledge`. Module 3 can then focus on grounded answers, abstaining when context is missing, and protected reservation writes.

In [ ]:
selected_module_3_retriever = search_hotel_knowledge
print('Selected for Module 3: workshop.hybrid_retrieval.search_hotel_knowledge')
driver.close()
print('Connection closed.')